# Multi-Period Optimization

Multi-period optimization models systems across multiple planning periods
(e.g. 2025, 2030, 2035). Each period shares the same timestep structure
but can have independent sizing and dispatch decisions.

This notebook demonstrates:

1. A gas boiler system with cost **and** CO₂ tracking across 3 periods
2. NPV discounting on costs via `period_weights_periodic` (CO₂ stays flat)
3. Which costs land in **temporal** vs **periodic** vs **once**

In [ ]:
from datetime import datetime

import pandas as pd

from fluxopt import Carrier, Converter, Effect, Flow, Port, Sizing, optimize

## System

A factory needs heat. A gas boiler is the only supply option,
with capacity to be optimized (sizing).

```
Gas Grid ──► [gas] ──► Boiler η=90% ──► [heat] ──► Factory
 0.05 €/kWh                                       50–100 kW
 0.2 kg CO₂/kWh
```

We plan across **3 periods** (2025, 2030, 2035), each representing 5 years.
The representative day has 4 timesteps.

In [ ]:
timesteps = [datetime(2025, 1, 15, h) for h in range(6, 10)]
periods = [2025, 2030, 2035]
demand_profile = [0.6, 1.0, 0.8, 0.5]  # relative to size=100 kW

## 1. Baseline: global period weights

With `periods=[2025, 2030, 2035]` the weights are inferred as `[5, 5, 5]`
(gap between labels, last gap repeated) — just like `dt` is inferred from timesteps.

In [ ]:
result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[
        Effect('cost', unit='EUR', is_objective=True),
        Effect('CO2', unit='kg'),
    ],
    ports=[
        Port(
            'gas_grid',
            imports=[
                Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.05, 'CO2': 0.2}),
            ],
        ),
        Port(
            'factory',
            exports=[
                Flow('heat', size=100, fixed_relative_profile=demand_profile),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow('gas', size=Sizing(50, 200, effects_per_size={'cost': 10})),
            thermal_flow=Flow('heat', size=200),
        ),
    ],
    periods=periods,
)

result.objective

### Where do costs end up?

Effects are split into three **domains**. Each captures a different kind of cost:

| Domain | What goes here | This example |
|---|---|---|
| **Temporal** | `effects_per_flow_hour` × flow × dt | Gas fuel cost (0.05 €/kWh), CO₂ (0.2 kg/kWh) |
| **Periodic** | `Sizing.effects_per_size` × size | Boiler sizing cost (10 €/kW) |
| **Once** | *(future: investment CAPEX)* | Currently zero — placeholder |

**Temporal** — per-timestep fuel costs (vary with dispatch):

In [ ]:
result.effects_temporal.sel(effect='cost').to_pandas()

**Periodic** — recurring sizing costs (boiler size × 10 €/kW):

In [ ]:
result.effects_periodic.sel(effect='cost').to_pandas()

**Once** — one-time costs (zero for now, future CAPEX placeholder):

In [ ]:
result.effects_once.sel(effect='cost').to_pandas()

**Total** = temporal_sum + periodic + once, per period.
The objective is `sum(period_weight[p] * total[p])` = 5 × total + 5 × total + 5 × total:

In [ ]:
result.effect_totals.sel(effect='cost').to_pandas()

## 2. NPV discounting on cost (CO₂ stays flat)

Future costs are worth less today. We apply NPV discount factors to the
cost effect via `Effect.period_weights_periodic`, while CO₂ keeps the
global period weights (physical quantity, no discounting).

With a 5% discount rate and 5-year periods, the annuity sum for each
period is `sum(1/(1+r)^y for y in range(5))` discounted to year 0:

In [ ]:
r = 0.05
period_years = [0, 5, 10]

npv_weights = [round(sum(1 / (1 + r) ** (y0 + y) for y in range(5)), 2) for y0 in period_years]

pd.DataFrame(
    {
        'global (duration)': [5, 5, 5],
        'cost (NPV)': npv_weights,
        'CO2 (no override)': [5, 5, 5],
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
result_npv = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[
        Effect('cost', unit='EUR', is_objective=True, period_weights_periodic=npv_weights),
        Effect('CO2', unit='kg'),
    ],
    ports=[
        Port(
            'gas_grid',
            imports=[
                Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.05, 'CO2': 0.2}),
            ],
        ),
        Port(
            'factory',
            exports=[
                Flow('heat', size=100, fixed_relative_profile=demand_profile),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow('gas', size=Sizing(50, 200, effects_per_size={'cost': 10})),
            thermal_flow=Flow('heat', size=200),
        ),
    ],
    periods=periods,
)

Same per-period costs, different objective contributions:

In [ ]:
cost_flat = result.effect_totals.sel(effect='cost').values
cost_npv = result_npv.effect_totals.sel(effect='cost').values

pd.DataFrame(
    {
        'cost/period': cost_flat.round(2),
        'weight (flat)': [5, 5, 5],
        'contribution (flat)': (cost_flat * 5).round(2),
        'weight (NPV)': npv_weights,
        'contribution (NPV)': (cost_npv * npv_weights).round(2),
    },
    index=pd.Index(periods, name='period'),
)

In [ ]:
pd.Series(
    {
        'Baseline (flat weights)': round(result.objective, 2),
        'NPV (discounted)': round(result_npv.objective, 2),
    },
    name='objective (EUR)',
)

## Recap: the three domains

56424\Phi_{k,p} = \underbrace{\sum_t \Phi^{\text{temporal}}_{k,t,p} \cdot w_t}_{\text{fuel, running, startup}} + \underbrace{\Phi^{\text{periodic}}_{k,p}}_{\text{sizing, O\&M}} + \underbrace{\Phi^{\text{once}}_{k,p}}_{\text{CAPEX (future)}}56424

The objective weights recurring and one-time costs **separately**:

56424\min \sum_p \omega^{\text{periodic}}_{k^*,p} \cdot (\text{temporal\_sum} + \text{periodic}) + \omega^{\text{once}}_{k^*,p} \cdot \text{once}56424

| Weight | Default | Override via | Use case |
|---|---|---|---|
| $\omega^{\text{periodic}}_{k,p}$ | Global `period_weights` | `Effect.period_weights_periodic` | NPV, annuity factors |
| $\omega^{\text{once}}_{k,p}$ | 1 (no scaling) | `Effect.period_weights_once` | Discount one-time investments |

The library doesn’t implement NPV — it provides **weight slots**.
You compute the discount factors and pass them in.